# Debug
Check the client offline, then inspect two separate debugging stories. Scripted responses below are test fixtures, not model predictions.

In [ ]:
import math
from tom_research.client import LLMClient, ModelConfig

def scripted_transport(request):
    return {'choices': [{'message': {'content': 'A'}, 'finish_reason': 'stop',
                         'logprobs': {'content': [{'top_logprobs': [
                             {'token': 'A', 'bytes': [65], 'logprob': math.log(0.4)}]}]}}],
            'usage': {'prompt_tokens': 10, 'completion_tokens': 1}}

client = LLMClient(ModelConfig(), transport=scripted_transport)
assert client.generate('Test request') == 'A'
assert abs(client.score_label('A) Likely. B) Unlikely.') - 0.4) < 1e-8
print('Scripted client checks passed. No server called.')

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
from tom_research.study import ROOT, load_study, prepare_pilot, provenance, track_paths
from tom_research.methods import LiveRunner, offline_runner
from tom_research.evaluation import run_pilot, summarize

TRACK = os.environ.get('TOM_TRACK', 'thoughttracing')
LIVE = False
config = json.loads((ROOT / 'configs/pilot.json').read_text())
source, manifest_path = track_paths(TRACK)
prepare_pilot(source, manifest_path, track=TRACK)
debug, pilot, manifest = load_study(source, manifest_path, track=TRACK)
methods = manifest['methods']
run_config = {**config, 'track': TRACK, 'methods': methods,
              'answer_format': manifest['answer_format'], 'mode': 'live' if LIVE else 'offline',
              'sources': provenance(track=TRACK)}
if LIVE:
    assert config['mode'] == 'live' and config['hardware'] and config['server_version'], 'Complete live preflight first'
    runner = LiveRunner(ModelConfig(**config['model_config']), config['item_seconds'])
else:
    runner = offline_runner
result_root = Path(os.environ.get('TOM_RESULTS_DIR', str(ROOT / 'results' / run_config['mode']))) / TRACK
records = run_pilot(debug, methods, runner, result_root / 'debug', run_config)
pd.DataFrame(summarize(records))

In [ ]:
for record in records:
    print(record['method'], record['id'], record['status'])
    if record['status'] == 'ok':
        print('Answer:', record['answer'])
        print('Trace:', record.get('trace', ''))
        print('Posterior:', record.get('posterior', []))
        print('Model:', record.get('model_record', {}))
    elif record['status'] == 'error':
        print(record['error'])

Before a live pilot, inspect the selected method's intermediate results and its plain baseline. Fix code with a failing test first; do not tune on the pilot questions.
The full solvers are not yet validated against a live Qwen server. ThoughtTracing currently uses Qwen for action labeling too, unlike the paper's GPT-4o helper. AutoToM's baseline uses an explicit exact-choice text prompt. Record these adaptations when comparing with published results.